# OOP Week 10 -- Testing with pytest

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-9
**Focus:** test organization, fixtures, parametrize, TDD

---

## Learning Objectives

1. Write test functions using pytest conventions
2. Use fixtures for shared test setup
3. Use parametrize for testing multiple cases
4. Organize tests by component
5. Practice Test-Driven Development (TDD)

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Setup: Classes to Test

In [ ]:
class CleaningStrategy:
    def should_keep(self, row):
        return True

class DropMissing(CleaningStrategy):
    def __init__(self, columns):
        self.columns = columns
    def should_keep(self, row):
        for col in self.columns:
            val = row.get(col)
            if val is None or (isinstance(val, str) and val.strip() == ""):
                return False
        return True

class DropOutOfRange(CleaningStrategy):
    def __init__(self, column, low, high):
        self.column = column
        self.low = low
        self.high = high
    def should_keep(self, row):
        val = row.get(self.column)
        if isinstance(val, (int, float)):
            return self.low <= val <= self.high
        return True

class ConfigurableCleaner:
    def __init__(self, strategies=None):
        self.strategies = strategies or []
    def add_strategy(self, strategy):
        self.strategies.append(strategy)
    def clean(self, data):
        return [row for row in data if all(s.should_keep(row) for s in self.strategies)]

print("Classes ready for testing.")

**Expected Output:**
```
Classes ready for testing.
```

---
## Section 1: pytest Basics

pytest is the standard Python testing framework. Key rules:

1. Test files start with `test_` (e.g., `test_cleaner.py`)
2. Test functions start with `test_` (e.g., `test_empty_input`)
3. Use plain `assert` statements (no special methods)
4. Run with `pytest tests/ -v`

We will write tests in this notebook, then show how they would look in actual test files.

In [ ]:
# Test 1: Cleaner handles empty input
def test_cleaner_empty_input():
    cleaner = ConfigurableCleaner()
    result = cleaner.clean([])
    assert result == []
    assert isinstance(result, list)

# Test 2: DropMissing drops None values
def test_drop_missing_none():
    s = DropMissing(["value"])
    assert s.should_keep({"value": 10}) == True
    assert s.should_keep({"value": None}) == False

# Test 3: DropOutOfRange works
def test_drop_out_of_range():
    s = DropOutOfRange("value", 0, 100)
    assert s.should_keep({"value": 50}) == True
    assert s.should_keep({"value": -10}) == False
    assert s.should_keep({"value": 200}) == False
    assert s.should_keep({"value": 0}) == True    # edge: exactly at min
    assert s.should_keep({"value": 100}) == True   # edge: exactly at max

# Test 4: Full pipeline
def test_full_cleaning():
    cleaner = ConfigurableCleaner([
        DropMissing(["value"]),
        DropOutOfRange("value", 0, 100)
    ])
    data = [
        {"value": 50},
        {"value": None},
        {"value": 200},
        {"value": 25},
    ]
    result = cleaner.clean(data)
    assert len(result) == 2
    assert result[0]["value"] == 50
    assert result[1]["value"] == 25


# Run tests
test_cleaner_empty_input()
print("[PASS] test_cleaner_empty_input")
test_drop_missing_none()
print("[PASS] test_drop_missing_none")
test_drop_out_of_range()
print("[PASS] test_drop_out_of_range")
test_full_cleaning()
print("[PASS] test_full_cleaning")
print()
print("All 4 tests passed!")

**Expected Output:**
```
[PASS] test_cleaner_empty_input
[PASS] test_drop_missing_none
[PASS] test_drop_out_of_range
[PASS] test_full_cleaning

All 4 tests passed!
```

---
## Section 2: Fixtures (Shared Test Setup)

A **fixture** is reusable setup code. In pytest, you use the `@pytest.fixture` decorator. In notebooks, we use plain functions.

In [ ]:
def make_sample_data():
    """Fixture: standard test dataset."""
    return [
        {"id": 1, "value": 25.0, "status": "ok"},
        {"id": 2, "value": 50.0, "status": "ok"},
        {"id": 3, "value": None, "status": "error"},
        {"id": 4, "value": 200.0, "status": "warning"},
        {"id": 5, "value": 75.0, "status": "ok"},
    ]

def make_cleaner():
    """Fixture: standard cleaner."""
    return ConfigurableCleaner([
        DropMissing(["value"]),
        DropOutOfRange("value", 0, 100),
    ])


def test_with_fixtures():
    data = make_sample_data()
    cleaner = make_cleaner()
    result = cleaner.clean(data)
    assert len(result) == 3  # ids 1, 2, 5
    values = [r["value"] for r in result]
    assert all(0 <= v <= 100 for v in values)

test_with_fixtures()
print("[PASS] test_with_fixtures")

**Expected Output:**
```
[PASS] test_with_fixtures
```

---
## Section 3: Parametrize (Testing Many Cases)

Instead of writing one test per case, you can test many inputs with one function. In pytest, use `@pytest.mark.parametrize`. In notebooks, we use a loop.

In [ ]:
# In a real pytest file, you would write:
# @pytest.mark.parametrize("value,expected", [
#     (50, True), (-10, False), (200, False), (0, True), (100, True)
# ])
# def test_range_check(value, expected):
#     s = DropOutOfRange("value", 0, 100)
#     assert s.should_keep({"value": value}) == expected

# In a notebook, we do it like this:
test_cases = [
    (50, True, "middle of range"),
    (-10, False, "below range"),
    (200, False, "above range"),
    (0, True, "at minimum"),
    (100, True, "at maximum"),
    (0.001, True, "just above min"),
    (99.999, True, "just below max"),
]

s = DropOutOfRange("value", 0, 100)
for value, expected, label in test_cases:
    result = s.should_keep({"value": value})
    status = "PASS" if result == expected else "FAIL"
    print("[" + status + "] value=" + str(value) + " -> " + str(result) + " (" + label + ")")
    assert result == expected, "Failed for " + label

print()
print("All", len(test_cases), "parametrized cases passed!")

**Expected Output:**
```
[PASS] value=50 -> True (middle of range)
[PASS] value=-10 -> False (below range)
[PASS] value=200 -> False (above range)
[PASS] value=0 -> True (at minimum)
[PASS] value=100 -> True (at maximum)
[PASS] value=0.001 -> True (just above min)
[PASS] value=99.999 -> True (just below max)

All 7 parametrized cases passed!
```

---
## Section 4: Test File Organization

In your project, tests should be organized like this:
```
tests/
    __init__.py
    test_data_source.py    # tests for DataSource
    test_cleaner.py        # tests for Cleaner
    test_analyzer.py       # tests for Analyzer
    test_plotter.py        # tests for Plotter
    test_reporter.py       # tests for Reporter
    test_exceptions.py     # tests for custom exceptions
    conftest.py            # shared fixtures
```

Run all tests: `pytest tests/ -v`
Run one file: `pytest tests/test_cleaner.py -v`
Run one test: `pytest tests/test_cleaner.py::test_empty_input -v`

---
### Try It!

Write 3 tests for a `MeanAnalyzer`:
1. Normal case: `[10, 20, 30]` -> mean is 20
2. Empty case: `[]` -> returns empty dict
3. Single value: `[42]` -> mean is 42

In [ ]:
# YOUR CODE HERE


---
## Mini-Quiz

In [ ]:
# Q1: What naming convention does pytest use for test functions?
# Answer: 

# Q2: What is a fixture?
# Answer: 

# Q3: What is parametrize useful for?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)